# fullstack — Databricks notebook

Runs end-to-end on a **Databricks** cluster: sample customers → Parquet → email token columns → Parquet.

**Import:** Upload this notebook or sync the repo via **Databricks Repos**.

**Setup:** Either install a wheel with widget **`pip_wheel_uri`** (see below), add **`fullstack`** as a cluster library, or `%pip install` from your Git artifact. Leave **`pip_wheel_uri`** empty once the package is installed.

**Widgets** set output locations (`dbfs:` or Unity Catalog **Volume** URIs).

Avoid calling **`spark.stop()`** on a shared cluster.


In [ ]:
import os
import subprocess
import sys

IS_DATABRICKS = bool(os.environ.get("DATABRICKS_RUNTIME_VERSION"))

if IS_DATABRICKS:
    dbutils.widgets.text("customers_parquet_uri", "dbfs:/tmp/fullstack/customers_parquet")
    dbutils.widgets.text("tokenized_parquet_uri", "dbfs:/tmp/fullstack/customers_tokenized")
    dbutils.widgets.text("pip_wheel_uri", "")


## Optional: install the `fullstack` package

If **`pip_wheel_uri`** is set (e.g. `/Workspace/Users/<user>/artifacts/fullstack-0.1.0-py3-none-any.whl`), the next cell installs it quietly. Leave empty if the wheel is already on the cluster.


In [ ]:
if IS_DATABRICKS:
    wheel = dbutils.widgets.get("pip_wheel_uri").strip()
    if wheel:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--quiet", wheel]
        )
    else:
        print("pip_wheel_uri empty — assuming `fullstack` is installed.")
else:
    print("Local: run from repo with `poetry install`.")


In [ ]:
import tempfile
from pathlib import Path

if IS_DATABRICKS:
    CUSTOMERS_URI = dbutils.widgets.get("customers_parquet_uri").rstrip("/")
    TOKENIZED_URI = dbutils.widgets.get("tokenized_parquet_uri").rstrip("/")
else:
    base = Path(tempfile.gettempdir()) / "fullstack_nb_demo"
    base.mkdir(parents=True, exist_ok=True)
    CUSTOMERS_URI = str(base / "customers_parquet")
    TOKENIZED_URI = str(base / "customers_tokenized")

print("Customers Parquet URI:", CUSTOMERS_URI)
print("Tokenized Parquet URI:", TOKENIZED_URI)


In [ ]:
if not IS_DATABRICKS:
    import sys
    from pathlib import Path

    here = Path.cwd()
    for cand in [here, *here.parents]:
        if (cand / "pyproject.toml").is_file():
            src = cand / "src"
            if str(src) not in sys.path:
                sys.path.insert(0, str(src))
            break
    else:
        raise RuntimeError(
            "Could not find pyproject.toml; run this notebook from the fullstack repo."
        )


In [ ]:
from pyspark.sql import functions as F

from fullstack.jobs.email_tokenization import with_email_token_columns
from fullstack.spark_session import get_spark

spark = get_spark(app_name="fullstack-databricks-notebook")
print("Spark:", spark.version)


In [ ]:
customers_df = spark.createDataFrame(
    [
        (1, "Alice", "alice@example.com"),
        (2, "Bob", "bob@example.com"),
        (3, "Carlos", "carlos@example.com"),
    ],
    ["customer_id", "name", "email"],
)

customers_enriched = customers_df.withColumn(
    "ingested_at",
    F.to_timestamp(F.lit("2026-01-01 09:30:00")),
).repartition(2)

customers_enriched.show(truncate=False)
customers_enriched.write.mode("overwrite").parquet(CUSTOMERS_URI)


In [ ]:
read_customers = spark.read.parquet(CUSTOMERS_URI)
tokenized = with_email_token_columns(read_customers)
tokenized.show(truncate=False)
tokenized.printSchema()
tokenized.write.mode("overwrite").parquet(TOKENIZED_URI)


In [ ]:
spark.read.parquet(TOKENIZED_URI).orderBy("customer_id").show(truncate=False)
